# Transformer 变体详解

本 Notebook 介绍 Transformer 的重要变体，重点关注高效注意力机制。

## 目录
1. [线性注意力（Linear Attention）](#1-线性注意力)
2. [滑动窗口注意力（Sliding Window Attention）](#2-滑动窗口注意力)
3. [Flash Attention 原理与使用](#3-flash-attention)
4. [Rotary Position Embedding (RoPE)](#4-rope)
5. [Grouped Query Attention (GQA)](#5-gqa)
6. [SWITCH Transformer / MoE 简化实现](#6-moe)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
## 1. 线性注意力

标准注意力的瓶颈在于 softmax(QK^T)，复杂度 $O(n^2)$。

线性注意力的核心思路：
$$\text{Attention}(Q, K, V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$$

通过先计算 $K^T V$（$d \times d$ 矩阵乘法），将复杂度降到 $O(n \cdot d^2)$。
当 $d < n$ 时，这就是线性复杂度。

In [ ]:
class LinearAttention(nn.Module):
    """线性注意力 (Katharopoulos et al., 2020)
    
    使用 ELU+1 作为特征映射函数 phi，避免 softmax 的 O(n^2) 瓶颈。
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def feature_map(self, x):
        """特征映射: ELU(x) + 1，保证非负"""
        return F.elu(x) + 1

    def forward(self, Q, K, V, mask=None):
        B, L, _ = Q.shape
        H = self.n_heads
        d_k = self.d_k

        Q = self.W_q(Q).view(B, L, H, d_k).transpose(1, 2)
        K = self.W_k(K).view(B, L, H, d_k).transpose(1, 2)
        V = self.W_v(V).view(B, L, H, d_k).transpose(1, 2)

        # 应用特征映射
        Q = self.feature_map(Q)
        K = self.feature_map(K)

        # 关键：先计算 K^T V，形状 [B, H, d_k, d_k]
        KV = torch.einsum('bhnd,bhne->bhde', K, V)

        # Q * (K^T V)，形状 [B, H, L, d_k]
        out = torch.einsum('bhnd,bhde->bhne', Q, KV)

        # 归一化：Q * (K^T 1)
        K_sum = K.sum(dim=2, keepdim=True)  # [B, H, 1, d_k]
        normalizer = torch.einsum('bhnd,bhkd->bhnk', Q, K_sum).squeeze(-1)  # [B, H, L]
        normalizer = normalizer.unsqueeze(-1) + 1e-6

        out = out / normalizer
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        out = self.W_o(out)
        return out


# 对比复杂度
import time

d_model, n_heads = 64, 4
standard_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True).to(device)
linear_attn = LinearAttention(d_model, n_heads).to(device)

print('序列长度 vs 计算时间:')
print(f'{"SeqLen":>8} {"Standard":>12} {"Linear":>12}')
print('-' * 35)

for seq_len in [64, 256, 1024, 4096]:
    x = torch.randn(4, seq_len, d_model, device=device)
    
    # Standard
    start = time.time()
    for _ in range(10):
        _ = standard_attn(x, x, x, need_weights=False)
    t_std = (time.time() - start) / 10 * 1000
    
    # Linear
    start = time.time()
    for _ in range(10):
        _ = linear_attn(x, x, x)
    t_lin = (time.time() - start) / 10 * 1000
    
    print(f'{seq_len:>8d} {t_std:>10.1f}ms {t_lin:>10.1f}ms')

---
## 2. 滑动窗口注意力（Sliding Window Attention）

Longformer / Mistral 的核心思路：每个 token 只关注局部窗口内的 token，复杂度 $O(n \cdot w)$，$w$ 为窗口大小。

In [ ]:
class SlidingWindowAttention(nn.Module):
    """滑动窗口注意力"""
    def __init__(self, d_model, n_heads, window_size, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.window_size = window_size
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, L, _ = x.shape
        H = self.n_heads
        d_k = self.d_k
        w = self.window_size

        Q = self.W_q(x).view(B, L, H, d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, H, d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, H, d_k).transpose(1, 2)

        # 构建滑动窗口掩码
        window_mask = torch.zeros(L, L, device=x.device)
        for i in range(L):
            left = max(0, i - w // 2)
            right = min(L, i + w // 2 + 1)
            window_mask[i, left:right] = 1

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        scores = scores.masked_fill(window_mask.unsqueeze(0).unsqueeze(0) == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.W_o(out), attn


# 可视化滑动窗口掩码
swa = SlidingWindowAttention(d_model=64, n_heads=4, window_size=5)
x = torch.randn(1, 16, 64)
_, attn = swa(x)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(attn[0, 0].detach().numpy(), cmap='Blues')
axes[0].set_title('Sliding Window Attention')

# 对比全局注意力
full_attn = torch.ones(16, 16)
axes[1].imshow(full_attn, cmap='Blues')
axes[1].set_title('Full Attention')
plt.show()
print(f'滑动窗口: 每个token只关注窗口内的 {5} 个token')
print(f'全局注意力: 每个token关注所有 {16} 个token')

---
## 3. Flash Attention

Flash Attention 不是一种新的注意力算法，而是一种**内存高效的注意力实现**。

核心思路：
- 分块计算（Tiling），避免在 HBM 中存储完整的 $n \times n$ 注意力矩阵
- 利用 SRAM（片上快速内存）进行中间计算
- 在线 Softmax 实现数值稳定

PyTorch 2.0+ 已内置支持：`F.scaled_dot_product_attention`

In [ ]:
# Flash Attention 使用示例 (PyTorch 2.0+)
print('=== PyTorch SDPA 后端 ===')
print(f'Flash Attention 可用: {torch.backends.cuda.flash_sdp_enabled() if torch.cuda.is_available() else "N/A (CPU)"}')

# 使用方式
if torch.cuda.is_available():
    Q = torch.randn(2, 8, 1024, 64, device='cuda', requires_grad=True)
    K = torch.randn(2, 8, 1024, 64, device='cuda', requires_grad=True)
    V = torch.randn(2, 8, 1024, 64, device='cuda', requires_grad=True)

    # 自动选择最优后端（Flash / Mem-Efficient / Math）
    with torch.nn.attention.sdpa_kernel(
        [torch.nn.attention.SDPBackend.FLASH_ATTENTION]
    ):
        out = F.scaled_dot_product_attention(Q, K, V)
    print(f'Flash Attention output: {out.shape}')
else:
    print('Flash Attention 需要 GPU，当前为 CPU 模式')

print('''
Flash Attention 的优势:
  1. 内存节省: O(n) 而非 O(n^2) 的显存占用
  2. 速度提升: 减少对 HBM 的读写次数
  3. 精确计算: 不是近似，结果与标准注意力完全一致
''')

---
## 4. Rotary Position Embedding (RoPE)

RoPE 是 LLaMA / Qwen / Mistral 等现代 LLM 使用的位置编码方案。

核心思路：通过旋转矩阵将**相对位置信息**编码到注意力计算中。

$$q_m = R_{\Theta,m} W_q x_m, \quad k_n = R_{\Theta,n} W_k x_n$$

内积 $q_m^T k_n$ 自然包含相对位置 $m - n$ 的信息。

In [ ]:
class RotaryEmbedding(nn.Module):
    """旋转位置编码"""
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[1]
        return (
            self.cos_cached[:seq_len].to(x.dtype),
            self.sin_cached[:seq_len].to(x.dtype),
        )


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_emb(x, cos, sin):
    # x: [B, H, L, d_k]
    cos = cos.unsqueeze(0).unsqueeze(0)  # [1, 1, L, d]
    sin = sin.unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin


# 可视化 RoPE
rope = RotaryEmbedding(dim=64)
x = torch.randn(1, 1, 32, 64)
cos, sin = rope(x, seq_len=32)

# 展示旋转效果：计算不同位置的 q^T k
q = torch.randn(1, 1, 32, 64)
k = torch.randn(1, 1, 32, 64)
q_rot = apply_rotary_emb(q, cos, sin)
k_rot = apply_rotary_emb(k, cos, sin)

# 以位置 16 为基准，看与各位置的相似度变化
pos = 16
before = (q[0, 0, pos] @ k[0, 0].T).detach().numpy()
after = (q_rot[0, 0, pos] @ k_rot[0, 0].T).detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(32), before)
axes[0].set_title('Before RoPE (no position info)')
axes[1].bar(range(32), after)
axes[1].set_title('After RoPE (relative position encoded)')
plt.show()
print('RoPE 使得相近位置的 token 获得更高的注意力分数')

---
## 5. Grouped Query Attention (GQA)

标准 MHA：每个 Head 有独立的 Q/K/V。
MQA (Multi-Query Attention)：所有 Head 共享 K/V。
GQA：将 Head 分成几组，组内共享 K/V，是 MHA 和 MQA 的折中。

被 LLaMA-2、Mistral 等广泛采用。

In [ ]:
class GroupedQueryAttention(nn.Module):
    """分组查询注意力 (GQA)
    
    n_kv_heads: K/V 的 head 数量 (<= n_heads)
    n_heads:     Q 的 head 数量
    """
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_groups = n_heads // n_kv_heads  # 每组 Q heads 数
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads * self.d_k)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, L, _ = x.shape
        d_k = self.d_k

        Q = self.W_q(x).view(B, L, self.n_heads, d_k).transpose(1, 2)     # [B, n_heads, L, d_k]
        K = self.W_k(x).view(B, L, self.n_kv_heads, d_k).transpose(1, 2)   # [B, n_kv_heads, L, d_k]
        V = self.W_v(x).view(B, L, self.n_kv_heads, d_k).transpose(1, 2)   # [B, n_kv_heads, L, d_k]

        # 扩展 K/V 以匹配 Q 的 head 数
        # [B, n_kv_heads, L, d_k] -> [B, n_heads, L, d_k]
        K = K.repeat_interleave(self.n_groups, dim=1)
        V = V.repeat_interleave(self.n_groups, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, L, -1)
        return self.W_o(out)


# 对比 MHA / GQA / MQA 的参数量
d_model = 512
configs = {
    'MHA (8 heads)': (8, 8),    # 标准: Q=8, KV=8
    'GQA (8Q, 2KV)': (8, 2),    # GQA:  Q=8, KV=2
    'GQA (8Q, 4KV)': (8, 4),    # GQA:  Q=8, KV=4
    'MQA (8Q, 1KV)': (8, 1),    # MQA:  Q=8, KV=1
}

print(f'{"配置":>18} {"KV参数":>8} {"总参数":>10}')
print('-' * 40)
for name, (n_h, n_kv) in configs.items():
    m = GroupedQueryAttention(d_model, n_h, n_kv)
    kv_params = sum(p.numel() for p in [m.W_k, m.W_v])
    total_params = sum(p.numel() for p in m.parameters())
    print(f'{name:>18} {kv_params:>8,} {total_params:>10,}')

print('\nKV 参数减少 → 推理时 KV-Cache 更小 → 生成更快')

---
## 6. MoE (Mixture of Experts)

MoE 的核心思路：大幅增加参数量但不等比增加计算量。

- 多个并行的 FFN（称为 Expert）
- Router（门控网络）为每个 token 选择 Top-K 个 Expert
- 只有被选中的 Expert 参与计算

In [ ]:
class Expert(nn.Module):
    """单个 FFN Expert"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class MoELayer(nn.Module):
    """MoE 层: Top-K 路由"""
    def __init__(self, d_model, d_ff, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(n_experts)
        ])
        self.router = nn.Linear(d_model, n_experts)

    def forward(self, x):
        B, L, D = x.shape
        x_flat = x.view(-1, D)  # [B*L, D]

        # Router 计算 gate 分数
        logits = self.router(x_flat)  # [B*L, n_experts]
        top_k_logits, top_k_indices = logits.topk(self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)

        # 初始化输出
        output = torch.zeros_like(x_flat)

        # 将 token 分配给对应的 Expert
        for i in range(self.top_k):
            expert_idx = top_k_indices[:, i]   # [B*L]
            weight = top_k_weights[:, i:i+1]    # [B*L, 1]

            for e in range(self.n_experts):
                mask = (expert_idx == e)
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_output = self.experts[e](expert_input)
                    output[mask] += weight[mask] * expert_output

        # 辅助损失（负载均衡）
        aux_loss = self._load_balance_loss(logits)
        return output.view(B, L, D), aux_loss

    def _load_balance_loss(self, logits):
        """鼓励各 Expert 被均匀使用"""
        n_tokens = logits.shape[0]
        # 每个 expert 被选中的概率
        probs = F.softmax(logits, dim=-1).mean(0)
        # 每个 expert 实际被分配的 token 比例
        _, indices = logits.topk(self.top_k, dim=-1)
        mask = torch.zeros_like(logits)
        mask.scatter_(1, indices, 1.0)
        fraction = mask.mean(0)
        # 负载均衡损失
        return n_tokens * (probs * fraction).sum()


# 验证
moe = MoELayer(d_model=64, d_ff=256, n_experts=4, top_k=2)
x = torch.randn(2, 16, 64)
out, aux = moe(x)
print(f'MoE Output: {out.shape}')
print(f'Load Balance Loss: {aux.item():.4f}')

dense_params = 64 * 256 * 2  # 单个 FFN
moe_params = sum(p.numel() for p in moe.experts.parameters())
print(f'\n单个 FFN 参数: {dense_params:,}')
print(f'4个 Expert 参数: {moe_params:,}')
print(f'但每次前向传播只激活 top-2 个 Expert，实际计算量 ≈ 2x FFN')

---
## Transformer 变体总结

| 变体 | 核心创新 | 复杂度 | 代表模型 |
|------|----------|--------|----------|
| 标准注意力 | 自注意力机制 | $O(n^2 d)$ | Transformer, BERT, GPT |
| 线性注意力 | 特征映射替代 softmax | $O(n d^2)$ | Linear Transformer |
| 滑动窗口 | 局部注意力 | $O(n w d)$ | Longformer, Mistral |
| Flash Attention | 分块计算节省显存 | $O(n^2 d)$ 但更高效 | PyTorch SDPA |
| RoPE | 旋转位置编码 | - | LLaMA, Qwen, Mistral |
| GQA | 分组 KV 共享 | 减少 KV-Cache | LLaMA-2, Mistral |
| MoE | 稀疏 Expert 路由 | 参数大但激活少 | Mixtral, Switch Transformer |

**下一节将介绍 Mamba —— 一种完全不同的序列建模范式（状态空间模型 SSM），将注意力复杂度降到 O(n)。**